# Vérifications finales sur Kaggle (GPU T4) — ~15 min

Version Kaggle de `final_checks_colab.ipynb`, pendant que Colab ne fournit pas de GPU. Elle fait **2 des 3 vérifications** :
1. **Tests** : relancer `pytest` et garder le log (l'ancien `results/logs/pytest.log` montre « 1 failed », d'avant le prefetcher).
2. **Représentation de l'entrée** : la config finale **avec et sans CLAHE**, 3 seeds, même session, **validation uniquement** (le test n'est pas touché).

Le **premier run mesuré** n'est **pas** fait ici : Kaggle a plus de cœurs CPU que Colab, le cache d'images s'y construit plus
vite, et la comparaison avec la baseline (mesurée sur le cœur unique de Colab) ne serait pas juste.

**Avant de lancer (panneau de droite, *Session options* / *Settings*) :**
- *Accelerator* : **GPU T4 x2** (pas P100 : les versions récentes de PyTorch ne le gèrent plus) ;
- *Internet* : **On** (il faut avoir vérifié son numéro de téléphone sur Kaggle).

Puis **Run All**. À la fin, `final_checks.zip` est dans l'onglet **Output** (`/kaggle/working`) : télécharge-le et donne-le à Claude.

## 0. Préparation (GPU, code, données) — ~5 min

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch; print("torch", torch.__version__, "| CUDA dispo :", torch.cuda.is_available())
assert torch.cuda.is_available(), "Pas de GPU : Settings > Accelerator > GPU T4 x2"

In [ ]:
import os, sys, json, subprocess
REPO_URL = "https://github.com/zeynabbelkasmi/gpu-dr-project"
REPO = "/tmp/gpu-dr-project"            # hors de /kaggle/working : données, cache et checkpoints ne vont pas dans Output
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
!git log --oneline -1
!pip -q install -r requirements.txt 2>&1 | tail -2
os.environ["FDR26_DATA"] = "/tmp/data"
!python scripts/download_data.py 2>&1 | tail -3
os.makedirs("results/week4", exist_ok=True)
!python -c "import sys; sys.path.insert(0,'scripts'); import record_env; record_env.main('results/week4/final_checks_environment.json')"

## 1. Tests (équivalence CPU/GPU, prefetcher) — ~1 min
Le log complet remplace `results/logs/pytest.log`. Attendu : **0 failed** (des tests CuPy peuvent être *skipped*).

In [ ]:
os.makedirs("results/logs", exist_ok=True)
!python -m pytest tests -v 2>&1 | tee results/logs/pytest.log | tail -6

## 2. Représentation de l'entrée : avec / sans CLAHE — ~6 min
Une seule différence avec `configs/optimized.yaml` : `clahe: false`. Même split, même modèle, 320 px, 5 époques,
seeds 42-44, **même session** pour les deux variantes. Comparaison sur la **validation** seulement.
On construit d'abord le cache d'images, pour que le premier run ne paie pas ce coût à lui seul.

In [ ]:
import yaml
cfg = yaml.safe_load(open("configs/optimized.yaml"))
cfg["preprocess"]["clahe"] = False
with open("configs/exp_clahe_off.yaml", "w") as f:
    f.write("# Input representation (Week 4 extra): configs/optimized.yaml with CLAHE switched off.\n"
            "# Only preprocess.clahe changes (split, model, 320 px, 5 epochs identical). Validation only.\n")
    yaml.safe_dump(cfg, f, sort_keys=False)
!diff configs/optimized.yaml configs/exp_clahe_off.yaml | grep "clahe"

import time
from src.data_loader import get_splits, build_cache
from src.paths import ROOT
base = yaml.safe_load(open("configs/optimized.yaml"))
t0 = time.perf_counter()
splits = get_splits(base)
for k in ("train", "val"):
    build_cache(splits[k], base["data"]["image_size"], str(ROOT / base["data"]["cache_dir"]))
print(f"cache d'images prêt ({time.perf_counter() - t0:.0f} s)")

In [ ]:
from src.train import main as train_main
from src.utils import gpu_warmup

OUT, CKPT = "results/week4/input_repr", "checkpoints/week4/input_repr"
VARIANTS = [("clahe_on", "configs/optimized.yaml"), ("clahe_off", "configs/exp_clahe_off.yaml")]
SEEDS = (42, 43, 44)
gpu_warmup("cuda")
for seed in SEEDS:                      # seed d'abord : une dérive du GPU touche les deux variantes pareil
    for v, cfg_path in VARIANTS:
        d = f"{v}_s{seed}"
        if os.path.exists(f"{OUT}/{d}/history.json"):
            continue
        print(f"\n[input_repr] {d} ({cfg_path})", flush=True)
        train_main(cfg_path, "cuda", run=d, seed=seed, out_root=OUT, ckpt_root=CKPT)

In [ ]:
import pandas as pd
rows = []
for v, _ in VARIANTS:
    for seed in SEEDS:
        h = json.load(open(f"{OUT}/{v}_s{seed}/history.json"))
        best = max(h["history"], key=lambda e: e["val_acc"])      # l'époque du checkpoint
        rows.append({"variant": v, "seed": seed, "setup_time_s": h["setup_time_s"],
                     "steady_epoch_s": h["steady_epoch_s"], "best_val_acc": h["best_val_acc"],
                     "val_f1_at_best": best["val_f1"], "best_epoch": best["epoch"]})
runs = pd.DataFrame(rows)
runs.to_csv("results/week4/input_repr_runs.csv", index=False)
cols = ["setup_time_s", "steady_epoch_s", "best_val_acc", "val_f1_at_best"]
agg = runs.groupby("variant", sort=False)[cols].agg(["mean", "std"])
agg.columns = [f"{c}_{s}" for c, s in agg.columns]
paired = runs.pivot(index="seed", columns="variant", values="val_f1_at_best")
agg["delta_val_f1_vs_on_mean"] = [0.0, (paired.clahe_off - paired.clahe_on).mean()]
agg["delta_val_f1_vs_on_std"] = [0.0, (paired.clahe_off - paired.clahe_on).std()]
agg.to_csv("results/week4/input_repr.csv")
print(agg.round(4).T)
print("\nF1 par seed :\n", paired.round(4))

## Fin : récupérer les résultats
Le zip apparaît dans l'onglet **Output** à droite (`/kaggle/working/final_checks.zip`) ; le lien ci-dessous marche aussi.

In [ ]:
import zipfile
from pathlib import Path
from IPython.display import FileLink, display
paths = [Path("results/logs/pytest.log"), Path("results/week4/input_repr.csv"),
         Path("results/week4/input_repr_runs.csv"), Path("results/week4/final_checks_environment.json"),
         Path("configs/exp_clahe_off.yaml"), *sorted(Path("results/week4/input_repr").rglob("*.json"))]
zip_path = "/kaggle/working/final_checks.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in paths:
        if p.exists():
            z.write(p, p.as_posix())
            print("+", p)
        else:
            print("MANQUANT :", p)
os.chdir("/kaggle/working")
display(FileLink("final_checks.zip"))